# FloodSense — Feature Engineering Pipeline
Processes all three source files and saves a model-ready Excel output.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [2]:
df   = pd.read_excel('floodsense_training_data_FINAL_CLEAN.xlsx')
elev = pd.read_excel('district_elevation_reference_FINAL_CLEAN.xlsx')
ndma = pd.read_excel('ndma_flood_impact_2022_FINAL_CLEAN.xlsx')

# Sort by district + ds_idx (sequential observation index per district)
df = df.sort_values(['district', 'ds_idx']).reset_index(drop=True)

print('Training data shape:', df.shape)
print('Districts:', df['district'].unique())
df.head()

Training data shape: (1365, 22)
Districts: ['Balochistan_District' 'KP_District' 'Sindh_District']


,date,district,evaporation,precipitation,pressure,soil_moisture,temperature,water_area_km2,wind_speed,humidity,...,temp_3day_avg,soil_3day_avg,day_of_year,month,year,is_monsoon,water_area_change,water_area_pct_change,ds_idx,flood_event
0,2023-01-01,Balochistan_District,-0.000230,0.000135,100639.5491,0.087776,16.712126,73.521606,0.625041,50.787158,...,16.370911,0.087946,1,1,2023,0,-178.235539,-0.707966,0,0
1,2023-01-01,Balochistan_District,-0.000230,0.000100,100639.5491,0.087776,16.712126,374.708945,0.625041,50.787158,...,16.429007,0.087849,1,1,2023,0,301.187339,4.096583,1,0
2,2023-01-02,Balochistan_District,-0.000215,0.000000,100800.0680,0.087691,16.070381,250.390562,2.556115,54.975463,...,16.498211,0.087747,2,1,2023,0,-124.318384,-0.331773,2,0
3,2023-01-03,Balochistan_District,-0.000217,0.000000,101029.1779,0.087694,15.405261,126.072178,2.904837,49.248183,...,16.062589,0.087720,3,1,2023,0,-124.318384,-0.496498,3,0
4,2023-01-04,Balochistan_District,-0.000214,0.000000,101026.6490,0.087631,14.635752,1.753794,2.287333,51.012060,...,15.370465,0.087672,4,1,2023,0,-124.318384,-0.986089,4,0


## 2. Merge Elevation Reference

In [3]:
elev_clean = elev[['district', 'avg_elevation_m']].copy()
df = df.merge(elev_clean, on='district', how='left')

print('Elevation merged:')
print(df[['district', 'avg_elevation_m']].drop_duplicates())

Elevation merged:
                 district  avg_elevation_m
0    Balochistan_District              610
455           KP_District              285
909        Sindh_District              109


## 3. Merge NDMA Regional Flood Severity

In [4]:
region_map = {
    'Sindh':       'Sindh_District',
    'KP':          'KP_District',
    'Balochistan': 'Balochistan_District',
}
ndma['district'] = ndma['region'].map(region_map)
ndma_rel = ndma[ndma['district'].notna()][['district', 'total_deaths', 'affected_population', 'total_houses_damaged']].copy()

ndma_rel['historical_severity_score'] = (
    ndma_rel['total_deaths']          / ndma_rel['total_deaths'].max()          * 0.4 +
    ndma_rel['affected_population']   / ndma_rel['affected_population'].max()   * 0.4 +
    ndma_rel['total_houses_damaged']  / ndma_rel['total_houses_damaged'].max()  * 0.2
)

df = df.merge(ndma_rel[['district', 'historical_severity_score']], on='district', how='left')
df['historical_severity_score'] = df['historical_severity_score'].fillna(0)

print('Severity scores per district:')
print(df[['district', 'historical_severity_score']].drop_duplicates())

Severity scores per district:
                 district  historical_severity_score
0    Balochistan_District                   0.436289
455           KP_District                   0.310668
909        Sindh_District                   1.000000


## 4. Clean Sentinel Values in `water_area_pct_change`
- `500` = sentinel for water appearing from zero (0 → nonzero) — cap at 99th percentile
- Values < −0.99 = near-total water loss — meaningful, keep as-is

In [5]:
PCT_CAP = df.loc[df['water_area_pct_change'] < 500, 'water_area_pct_change'].quantile(0.99)
df['water_area_pct_change_clean']  = df['water_area_pct_change'].clip(upper=PCT_CAP)
df['water_area_new_appearance']    = (df['water_area_pct_change'] == 500).astype(int)

print(f'Sentinel 500 capped at: {PCT_CAP:.2f}')
print(f'New-appearance flags:   {df["water_area_new_appearance"].sum()}')

Sentinel 500 capped at: 335.45
New-appearance flags:   16


## 5. Precipitation Features

In [6]:
# Log-transform: precipitation is extremely right-skewed
df['precip_log1p']       = np.log1p(df['precipitation'])
df['precip_3day_log1p']  = np.log1p(df['precip_3day_avg'])
df['precip_7day_log1p']  = np.log1p(df['precip_7day_avg'])

# Acceleration: positive = rain intensifying; negative = easing
df['precip_acceleration'] = df['precip_3day_avg'] - df['precip_7day_avg']

# Is it actually raining today?
df['rain_today'] = (df['precipitation'] > 0.01).astype(int)

# 3-to-7 day ratio: >1 means recent rain exceeds weekly baseline
df['precip_3_to_7_ratio'] = np.where(
    df['precip_7day_avg'] > 0,
    df['precip_3day_avg'] / df['precip_7day_avg'],
    1.0
).clip(0, 10)

## 6. Soil Moisture Features

In [7]:
# Recent saturation gain vs 3-day baseline
df['soil_moisture_trend'] = df['soil_moisture'] - df['soil_3day_avg']

# Wet soil + heavy rain = amplified flood risk
df['rain_soil_interaction'] = df['precip_log1p'] * df['soil_moisture']

# Low elevation + saturated soil = high risk
elev_min = df['avg_elevation_m'].min()
elev_max = df['avg_elevation_m'].max()
df['elev_norm']          = (df['avg_elevation_m'] - elev_min) / (elev_max - elev_min + 1e-9)
df['soil_elevation_risk'] = df['soil_moisture'] * (1 - df['elev_norm'])

## 7. Water Extent Features

In [8]:
df['water_area_log1p']            = np.log1p(df['water_area_km2'])
df['water_area_change_abs']       = df['water_area_change'].abs()

# Preserves direction + compresses scale
df['water_area_change_signed_log'] = (
    np.sign(df['water_area_change']) * np.log1p(df['water_area_change'].abs())
)

## 8. Atmospheric Features

In [9]:
# Convective instability proxy
df['heat_humidity_index'] = df['temperature'] * df['humidity'] / 100.0

# Negative anomaly = low pressure = approaching storm
pressure_mean          = df.groupby('district')['pressure'].transform('mean')
df['pressure_anomaly'] = df['pressure'] - pressure_mean

# Negative evaporation = condensation / moisture input
df['evap_negative_flag'] = (df['evaporation'] < 0).astype(int)
df['evaporation_abs']    = df['evaporation'].abs()

## 9. Temporal / Cyclical Features

In [10]:
# Cyclical encoding: Jan(1) and Dec(12) are treated as adjacent
df['month_sin'] = np.sin(2 * np.pi * df['month']      / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month']      / 12)
df['doy_sin']   = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos']   = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Monsoon interaction: features behave differently in monsoon season
df['precip_x_monsoon']      = df['precip_log1p']      * df['is_monsoon']
df['soil_x_monsoon']        = df['soil_moisture']     * df['is_monsoon']
df['water_area_x_monsoon']  = df['water_area_log1p']  * df['is_monsoon']

# Fractional position in year (0–1)
df['season_position'] = df['day_of_year'] / 365.0

## 10. Flood History Features
`ds_idx` is a sequential observation index per district (time-ordered). 
We use it to compute lag-based flood memory features with no data leakage (shift(1)).

In [11]:
def rolling_flood_rate(group, window):
    return group['flood_event'].rolling(window=window, min_periods=1).mean().shift(1).fillna(0)

df['flood_rate_7d']  = df.groupby('district', group_keys=False).apply(lambda g: rolling_flood_rate(g, 7))
df['flood_rate_14d'] = df.groupby('district', group_keys=False).apply(lambda g: rolling_flood_rate(g, 14))

def days_since_last_flood(group):
    result = []
    last_flood = np.nan
    for idx, row in group.iterrows():
        result.append(np.nan if np.isnan(last_flood) else row['ds_idx'] - last_flood)
        if row['flood_event'] == 1:
            last_flood = row['ds_idx']
    return pd.Series(result, index=group.index)

df['days_since_flood'] = df.groupby('district', group_keys=False).apply(days_since_last_flood)
df['days_since_flood'] = df['days_since_flood'].fillna(999).clip(upper=365)

## 11. District Encoding

In [12]:
# Label encode for tree-based models
district_codes = {'Sindh_District': 0, 'Balochistan_District': 1, 'KP_District': 2}
df['district_code'] = df['district'].map(district_codes)

# One-hot for linear models (drop_first avoids dummy trap)
dummies = pd.get_dummies(df['district'], prefix='district', drop_first=True).astype(int)
df = pd.concat([df, dummies], axis=1)

## 12. Composite Flood Risk Score
A hand-crafted interpretable index — useful as a meta-feature or sanity check. **Not the target.**

In [13]:
def minmax_scale(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df['flood_risk_score'] = (
    0.25 * minmax_scale(df['precip_log1p']) +
    0.20 * minmax_scale(df['soil_moisture']) +
    0.20 * minmax_scale(df['water_area_log1p']) +
    0.10 * minmax_scale(df['water_area_change'].clip(lower=0)) +
    0.10 * (1 - minmax_scale(df['avg_elevation_m'])) +
    0.10 * minmax_scale(df['flood_rate_7d']) +
    0.05 * df['is_monsoon']
)

## 13. Build Final Model-Ready Dataset

In [14]:
# Drop columns not suitable as model features
# - date: replaced by temporal encodings
# - district: replaced by district_code + dummies
# - elev_norm: intermediate, rolled into soil_elevation_risk
# - water_area_pct_change: replaced by cleaned version
EXCLUDE = ['date', 'district', 'elev_norm', 'water_area_pct_change']

feature_cols = [c for c in df.columns if c not in EXCLUDE and c != 'flood_event']
target_col   = 'flood_event'

df_model = df[feature_cols + [target_col]].copy()

# Sanity checks
assert df_model.select_dtypes(include='object').empty,   'Object columns remain!'
assert not np.isinf(df_model.select_dtypes(include='number').values).any(), 'Inf values found!'
assert df_model.isnull().sum().sum() == 0,               'Null values found!'

print(f'Model-ready dataset: {df_model.shape[0]} rows x {df_model.shape[1]} columns')
print(f'Features: {len(feature_cols)}  |  Target: flood_event')
print(f'Class balance  ->  0: {(df_model[target_col]==0).sum()}  |  1: {(df_model[target_col]==1).sum()}')
df_model.head()

Model-ready dataset: 1365 rows x 54 columns
Features: 53  |  Target: flood_event
Class balance  ->  0: 923  |  1: 442


,evaporation,precipitation,pressure,soil_moisture,temperature,water_area_km2,wind_speed,humidity,precip_3day_avg,precip_7day_avg,...,water_area_x_monsoon,season_position,flood_rate_7d,flood_rate_14d,days_since_flood,district_code,district_KP_District,district_Sindh_District,flood_risk_score,flood_event
0,-0.000230,0.000135,100639.5491,0.087776,16.712126,73.521606,0.625041,50.787158,0.000159,0.004001,...,0.0,0.002740,0.0,0.0,365.0,1,0,0,0.112836,0
1,-0.000230,0.000100,100639.5491,0.087776,16.712126,374.708945,0.625041,50.787158,0.000204,0.002868,...,0.0,0.002740,0.0,0.0,365.0,1,0,0,0.169773,0
2,-0.000215,0.000000,100800.0680,0.087691,16.070381,250.390562,2.556115,54.975463,0.000090,0.001540,...,0.0,0.005479,0.0,0.0,365.0,1,0,0,0.143224,0
3,-0.000217,0.000000,101029.1779,0.087694,15.405261,126.072178,2.904837,49.248183,0.000045,0.000211,...,0.0,0.008219,0.0,0.0,365.0,1,0,0,0.126144,0
4,-0.000214,0.000000,101026.6490,0.087631,14.635752,1.753794,2.287333,51.012060,0.000000,0.000087,...,0.0,0.010959,0.0,0.0,365.0,1,0,0,0.030170,0


## 14. Save All Outputs to Excel

In [15]:
# ── File 1: Model-ready engineered features ───────────────────────────────────
with pd.ExcelWriter('floodsense_features_engineered.xlsx', engine='openpyxl') as writer:

    # Sheet 1: all features + target
    df_model.to_excel(writer, sheet_name='model_ready', index=False)

    # Sheet 2: feature dictionary
    feature_dict = {
        'precipitation':                'Raw daily precipitation (mm)',
        'precip_3day_avg':              'Raw 3-day rolling avg precipitation',
        'precip_7day_avg':              'Raw 7-day rolling avg precipitation',
        'soil_moisture':                'Raw soil moisture (volumetric fraction)',
        'soil_3day_avg':                'Raw 3-day rolling avg soil moisture',
        'water_area_km2':               'Raw surface water extent (km²)',
        'water_area_change':            'Raw change in water extent vs prev obs',
        'water_area_pct_change_clean':  'Cleaned pct change (sentinel 500 capped at 99th pct)',
        'water_area_new_appearance':    'Binary: 1 if original value was 500 sentinel',
        'temperature':                  'Raw daily temperature (°C)',
        'humidity':                     'Raw relative humidity (%)',
        'pressure':                     'Raw atmospheric pressure (Pa)',
        'evaporation':                  'Raw evaporation (negative = condensation)',
        'month':                        'Calendar month (1–12)',
        'day_of_year':                  'Day of year (1–365)',
        'is_monsoon':                   'Binary: 1 if month in [6,7,8,9]',
        'ds_idx':                       'Sequential obs index per district (0-based, time-ordered)',
        'avg_elevation_m':              'District avg elevation (m) — merged from reference file',
        'historical_severity_score':    'Normalised NDMA 2022 severity score per district',
        'precip_log1p':                 'log(1 + precipitation) — compresses right-skew',
        'precip_3day_log1p':            'log(1 + precip_3day_avg)',
        'precip_7day_log1p':            'log(1 + precip_7day_avg)',
        'precip_acceleration':          'precip_3day_avg - precip_7day_avg (rain intensifying signal)',
        'rain_today':                   'Binary: precipitation > 0.01 mm',
        'precip_3_to_7_ratio':          'Ratio of 3-day to 7-day avg (clipped 0–10)',
        'soil_moisture_trend':          'soil_moisture - soil_3day_avg (recent saturation gain)',
        'rain_soil_interaction':        'precip_log1p × soil_moisture — combined flood load',
        'soil_elevation_risk':          'soil_moisture × (1 - normalised elevation)',
        'water_area_log1p':             'log(1 + water_area_km2)',
        'water_area_change_abs':        'Absolute value of water_area_change',
        'water_area_change_signed_log': 'sign(change) × log(1+|change|) — direction + scale',
        'heat_humidity_index':          'temperature × humidity / 100 — convective instability',
        'pressure_anomaly':             'pressure - district mean pressure (negative = storm)',
        'evap_negative_flag':           'Binary: 1 if evaporation < 0',
        'evaporation_abs':              'Absolute evaporation magnitude',
        'month_sin':                    'Cyclical sin encoding of month',
        'month_cos':                    'Cyclical cos encoding of month',
        'doy_sin':                      'Cyclical sin encoding of day_of_year',
        'doy_cos':                      'Cyclical cos encoding of day_of_year',
        'precip_x_monsoon':             'precip_log1p × is_monsoon',
        'soil_x_monsoon':               'soil_moisture × is_monsoon',
        'water_area_x_monsoon':         'water_area_log1p × is_monsoon',
        'season_position':              'day_of_year / 365 — fractional year position',
        'flood_rate_7d':                'Rolling flood frequency over last 7 obs (lag-1, no leakage)',
        'flood_rate_14d':               'Rolling flood frequency over last 14 obs (lag-1, no leakage)',
        'days_since_flood':             'Obs steps since last flood per district (999 if never seen)',
        'district_code':                'Label encode: Sindh=0, Balochistan=1, KP=2',
        'district_KP_District':         'One-hot: 1 if KP_District',
        'district_Sindh_District':      'One-hot: 1 if Sindh_District',
        'flood_risk_score':             'Composite meta-feature (precip+soil+water+elev+history)',
        'year':                         'Calendar year — use for train/val split',
        'flood_event':                  'TARGET — 1 = flood event, 0 = no flood',
    }
    dict_df = pd.DataFrame([
        {'feature': k, 'description': v,
         'type': 'Engineered' if k not in [
             'precipitation','precip_3day_avg','precip_7day_avg','soil_moisture',
             'soil_3day_avg','water_area_km2','water_area_change','temperature',
             'humidity','pressure','evaporation','month','day_of_year',
             'is_monsoon','ds_idx','avg_elevation_m','year','flood_event',
             'wind_speed','temp_3day_avg'
         ] else 'Raw/Merged'}
        for k, v in feature_dict.items() if k in df_model.columns or k == 'flood_event'
    ])
    dict_df.to_excel(writer, sheet_name='feature_dictionary', index=False)

    # Sheet 3: summary statistics
    df_model.describe().T.to_excel(writer, sheet_name='feature_statistics')

print('Saved: floodsense_features_engineered.xlsx')

Saved: floodsense_features_engineered.xlsx


In [16]:
# ── File 2: Enriched elevation reference ─────────────────────────────────────
elev_enriched = elev.copy()
elev_enriched['elev_norm']             = (elev_enriched['avg_elevation_m'] - elev_enriched['avg_elevation_m'].min()) / \
                                          (elev_enriched['avg_elevation_m'].max() - elev_enriched['avg_elevation_m'].min() + 1e-9)
elev_enriched['flood_risk_elev_class'] = pd.cut(
    elev_enriched['avg_elevation_m'],
    bins=[-np.inf, 200, 400, np.inf],
    labels=['High Risk (Low Elev)', 'Medium Risk', 'Low Risk (High Elev)']
)
elev_enriched['district_code'] = elev_enriched['district'].map(district_codes)

elev_enriched.to_excel('district_elevation_reference_enriched.xlsx', index=False)
print('Saved: district_elevation_reference_enriched.xlsx')
elev_enriched

Saved: district_elevation_reference_enriched.xlsx


,district,avg_elevation_m,terrain_type,elev_norm,flood_risk_elev_class,district_code
0,Sindh_District,109,Flat floodplain — Dadu monitoring station elev...,0.000000,High Risk (Low Elev),0
1,Balochistan_District,610,Semi-arid plateau — Balochistan plateau averag...,1.000000,Low Risk (High Elev),1
2,KP_District,285,River valley — Nowshera city elevation on Kabu...,0.351297,Medium Risk,2


In [17]:
# ── File 3: Enriched NDMA impact reference ────────────────────────────────────
ndma_enriched = ndma.copy()

# Normalised severity components
ndma_enriched['severity_deaths_norm']     = ndma_enriched['total_deaths']         / ndma_enriched['total_deaths'].max()
ndma_enriched['severity_population_norm'] = ndma_enriched['affected_population']  / ndma_enriched['affected_population'].max()
ndma_enriched['severity_houses_norm']     = ndma_enriched['total_houses_damaged'] / ndma_enriched['total_houses_damaged'].max()

# Composite score (same weights used in training merge)
ndma_enriched['historical_severity_score'] = (
    ndma_enriched['severity_deaths_norm']     * 0.4 +
    ndma_enriched['severity_population_norm'] * 0.4 +
    ndma_enriched['severity_houses_norm']     * 0.2
)

# Injury rate (injured per death, where deaths > 0)
ndma_enriched['injury_to_death_ratio'] = np.where(
    ndma_enriched['total_deaths'] > 0,
    ndma_enriched['total_injured'] / ndma_enriched['total_deaths'],
    np.nan
)

# Houses damaged per affected person
ndma_enriched['houses_per_capita'] = np.where(
    ndma_enriched['affected_population'] > 0,
    ndma_enriched['total_houses_damaged'] / ndma_enriched['affected_population'],
    np.nan
)

ndma_enriched.to_excel('ndma_flood_impact_2022_enriched.xlsx', index=False)
print('Saved: ndma_flood_impact_2022_enriched.xlsx')
ndma_enriched

Saved: ndma_flood_impact_2022_enriched.xlsx


,region,total_deaths,m_d,f_d,c_d,total_injured,m_i,f_i,c_i,roads_damaged_km,...,livestock_damaged,affected_population,data_notes,district,severity_deaths_norm,severity_population_norm,severity_houses_norm,historical_severity_score,injury_to_death_ratio,houses_per_capita
0,AJ&K,48,31,17,0,24,15,9,0,0,...,792,53700,NaN,NaN,0.070796,0.003687,0.000319,0.029857,0.500000,0.010205
1,Balochistan,299,136,73,90,181,92,40,49,1850,...,500000,9182616,NaN,Balochistan_District,0.441003,0.630511,0.038420,0.436289,0.605351,0.007187
2,GB,22,5,11,6,6,3,0,3,16,...,0,51500,NaN,NaN,0.032448,0.003536,0.000705,0.014535,0.272727,0.023515
3,ICT,1,1,0,0,0,0,0,0,0,...,0,0,NaN,NaN,0.001475,0.000000,0.000000,0.000590,0.000000,NaN
4,KP,306,149,41,116,369,156,79,134,1575,...,21328,4350490,NaN,KP_District,0.451327,0.298720,0.053245,0.310668,1.205882,0.021024
5,Punjab,191,94,47,50,3858,2173,113,572,896,...,205106,4844253,WARNING: m_i+f_i+c_i=2858 but total_injured=38...,NaN,0.281711,0.332624,0.038987,0.253531,20.198953,0.013825
6,Sindh,678,262,126,290,8422,2954,2211,3247,8398,...,216683,14563770,NOTE: m_i+f_i+c_i=8412 but total_injured=8422 ...,Sindh_District,1.000000,1.000000,1.000000,1.000000,12.421829,0.117949


## 15. Final Verification

In [18]:
print('=== FINAL CHECKS ===')
print(f'Shape         : {df_model.shape}')
print(f'Null values   : {df_model.isnull().sum().sum()}')
print(f'Inf values    : {np.isinf(df_model.select_dtypes(include="number")).sum().sum()}')
print(f'Object cols   : {df_model.select_dtypes(include="object").columns.tolist()}')
print(f'Target (0/1)  : {(df_model.flood_event==0).sum()} / {(df_model.flood_event==1).sum()}')

print('\nTop 10 features by correlation with flood_event:')
corrs = df_model.select_dtypes(include='number').corr()['flood_event'].abs().sort_values(ascending=False)
print(corrs.head(11).to_string())

print('\nAll outputs saved successfully.')

=== FINAL CHECKS ===
Shape         : (1365, 54)
Null values   : 0
Inf values    : 0
Object cols   : []
Target (0/1)  : 923 / 442

Top 10 features by correlation with flood_event:
flood_event                     1.000000
water_area_km2                  0.812613
water_area_log1p                0.529415
water_area_change_abs           0.303226
flood_risk_score                0.303041
water_area_x_monsoon            0.199308
water_area_change               0.179766
month_cos                       0.172869
water_area_pct_change_clean     0.159198
doy_cos                         0.158613
water_area_change_signed_log    0.138199

All outputs saved successfully.
